In [1]:
import os, sys, time, numpy as np, struct, cv2
from pynq import allocate
from pynq_dpu import DpuOverlay
import vart, xir

# --- 1. 配置路径与环境 ---
WORK_DIR = '/home/xilinx/jupyter_notebooks/duxu/pynq_vqvae'
SUB_DIR = os.path.join(WORK_DIR, 'pl_vq_31ms_260429')
BIT_PATH = os.path.join(SUB_DIR, 'dpu.bit')
PRE_DIR = os.path.join(SUB_DIR, 'imgs_preprocessed')
RESULT_DIR = os.path.join(SUB_DIR, 'results_pingpong')

if not os.path.exists(RESULT_DIR): os.makedirs(RESULT_DIR)

# 加载模型路径
ENC_PATH = os.path.join(WORK_DIR, 'xmodel/encoder_zcu111_700x500_old.xmodel')
DEC_PATH = os.path.join(WORK_DIR, 'xmodel/decoder_zcu111_700x500_old.xmodel')

# 加载位流
print("📡 Loading Overlay...")
overlay = DpuOverlay(BIT_PATH)
vq_ip = overlay.vq_accel_1

# --- 2. 硬件 Buffer 准备 (Ping-Pong 模式) ---
num_vectors, dim = 125 * 175, 64

# 分配两个输入 Buffer，强制硬件地址切换
vq_in_z_ping = allocate(shape=(num_vectors, dim), dtype=np.int8, cacheable=0)
vq_in_z_pong = allocate(shape=(num_vectors, dim), dtype=np.int8, cacheable=0)
vq_in_bufs = [vq_in_z_ping, vq_in_z_pong]

# Codebook 和输出 Buffer
vq_in_cb = allocate(shape=(512, 64), dtype=np.float32, cacheable=0)
vq_out_zq = allocate(shape=(num_vectors, dim), dtype=np.int8, cacheable=0)

# 加载 Codebook 数据
vq_in_cb[:] = np.load(os.path.join(WORK_DIR, 'codebook.npy')).astype(np.float32)
vq_in_cb.sync_to_device()

def setup_hardware(active_in_buf):
    """ 更新 VQ 加速器寄存器 """
    # 写入当前的输入 Buffer 地址
    vq_ip.register_map.in_z_1 = active_in_buf.device_address & 0xFFFFFFFF
    vq_ip.register_map.in_z_2 = active_in_buf.device_address >> 32
    
    # 写入固定的 Codebook 和输出地址
    vq_ip.register_map.in_codebook_1 = vq_in_cb.device_address & 0xFFFFFFFF
    vq_ip.register_map.in_codebook_2 = vq_in_cb.device_address >> 32
    vq_ip.register_map.out_z_q_1 = vq_out_zq.device_address & 0xFFFFFFFF
    vq_ip.register_map.out_z_q_2 = vq_out_zq.device_address >> 32
    
    # 写入 Scale 参数 (MMIO 方式)
    # f_enc = struct.unpack('<I', struct.pack('<f', 0.015625))[0]
    # 故意把 enc_out_scale (0.015625) 改成一个巨大的数，比如 100.0
    f_enc = struct.unpack('<I', struct.pack('<f', 100.0))[0]
    vq_ip.mmio.write(0x34, f_enc)
    
    f_dec = struct.unpack('<I', struct.pack('<f', 32.0))[0]
    vq_ip.mmio.write(0x34, f_enc)
    vq_ip.mmio.write(0x3c, f_dec)

# --- 3. 串行处理循环 ---
data_files = sorted([f for f in os.listdir(PRE_DIR) if f.endswith('.npy')])
print(f"🚀 Starting Ping-Pong Pipeline: {len(data_files)} images")

start_time = time.time()

for i, f in enumerate(data_files):
    # --- STEP A: DPU Encoder (每帧重建以释放总线) ---
    input_data = np.load(os.path.join(PRE_DIR, f))
    
    graph_e = xir.Graph.deserialize(ENC_PATH)
    runner_e = vart.Runner.create_runner(graph_e.get_root_subgraph().toposort_child_subgraph()[1], "run")
    
    in_buf = [np.ascontiguousarray(input_data[np.newaxis])]
    out_buf = [np.empty((1, 125, 175, 64), dtype=np.int8, order='C')]
    
    runner_e.wait(runner_e.execute_async(in_buf, out_buf))
    dpu_feat = out_buf[0].copy()
    
    # 销毁 DPU 资源
    del runner_e
    del graph_e

    # --- STEP B: VQ 硬件 (Ping-Pong 切换) ---
    # 1. 切换物理 Buffer
    active_buf = vq_in_bufs[i % 2]
    active_buf[:] = dpu_feat.reshape(num_vectors, dim)
    active_buf.sync_to_device()
    
    # 2. 硬件配置与启动
    setup_hardware(active_buf)
    
    vq_ip.register_map.CTRL.AP_START = 1
    # 轮询直到完成
    while not vq_ip.register_map.CTRL.AP_DONE:
        pass
    
    # 3. 取回结果
    vq_out_zq.sync_from_device()
    zq_snap = np.array(vq_out_zq, copy=True)
    
    # 打印验证信息
    print(f" 🎞️ [{i}] {f[:12]}... | Mean: {np.mean(zq_snap):.4f} | Addr: {hex(active_buf.device_address)}")

    # --- STEP C: 保存验证图片 (前 5 张) ---
    if i < 5:
        graph_d = xir.Graph.deserialize(DEC_PATH)
        runner_d = vart.Runner.create_runner(graph_d.get_root_subgraph().toposort_child_subgraph()[1], "run")
        
        dec_out = [np.empty((1, 500, 700, 3), dtype=np.int8, order='C')]
        runner_d.wait(runner_d.execute_async([zq_snap.reshape(1, 125, 175, 64)], dec_out))
        
        # 反量化反归一化 (dec_out_scale=0.007812)
        recon = (dec_out[0].astype(np.float32) * 0.007812 * 0.5 + 0.5).clip(0, 1)
        recon_img = (recon[0] * 255).astype(np.uint8)
        
        # 保存图片
        cv2.imwrite(os.path.join(RESULT_DIR, f'recon_pingpong_{i}.png'), 
                    cv2.cvtColor(recon_img, cv2.COLOR_RGB2BGR))
        
        del runner_d
        del graph_d

# --- 4. 总结 ---
total_dur = (time.time() - start_time) * 1000
print(f"\n✅ Pipeline Finished.")
print(f"Average Latency: {total_dur/len(data_files):.2f} ms/image")
print(f"Results saved in: {RESULT_DIR}")

📡 Loading Overlay...


🚀 Starting Ping-Pong Pipeline: 10 images
 🎞️ [0] 34b5c7611f5c... | Mean: 0.0856 | Addr: 0x78400000
 🎞️ [1] 90a3709413f8... | Mean: 0.0856 | Addr: 0x78600000
 🎞️ [2] DSCF8027.JPG... | Mean: 0.0856 | Addr: 0x78400000
 🎞️ [3] DSCF8031.JPG... | Mean: 0.0856 | Addr: 0x78600000
 🎞️ [4] DSCF8032.JPG... | Mean: 0.0856 | Addr: 0x78400000
 🎞️ [5] DSCF8049.JPG... | Mean: 0.0856 | Addr: 0x78600000
 🎞️ [6] GA0A5018.JPG... | Mean: 0.0856 | Addr: 0x78400000
 🎞️ [7] test2.jpg.np... | Mean: 0.0856 | Addr: 0x78600000
 🎞️ [8] wallhaven-l3... | Mean: 0.0856 | Addr: 0x78400000
 🎞️ [9] wallroom-192... | Mean: 0.0856 | Addr: 0x78600000

✅ Pipeline Finished.
Average Latency: 1420.87 ms/image
Results saved in: /home/xilinx/jupyter_notebooks/duxu/pynq_vqvae/pl_vq_31ms_260429/results_pingpong
